# ASR Baseline (Whisper) – Zeroth Korean

이 노트북은 Zeroth Korean 테스트 세트를 대상으로 **Whisper ASR 베이스라인 성능(WER / CER)**을 빠르게 확인하기 위한 실험용 노트북입니다.

- 입력 오디오: `/home/data/data/zeroth/test_data_01/**/**/*.flac` (하위 디렉터리를 포함한 모든 flac 파일)
- 정답 스크립트: `/home/data/data/zeroth/test_data_01/refs.csv` (파일명 기준으로 오디오와 매칭)


In [ ]:
import os, glob, time
import pandas as pd
import torch
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from jiwer import wer, cer
from pathlib import Path

In [ ]:
HOME = Path.home()

DATA_AUDIO_DIR = "/home/data/data/zeroth/test_data_01"
REF_CSV = "/home/data/data/zeroth/test_data_01/refs.csv"

MODEL_ID = "openai/whisper-base"

LANGUAGE = "korean"
TASK = "transcribe"

MAX_FILES = 50  # 빠른 확인용

print("MODEL_ID:", MODEL_ID)
print("DATA_AUDIO_DIR:", DATA_AUDIO_DIR)
print("REF_CSV:", REF_CSV)

In [ ]:
import os, torch

if torch.cuda.is_available():
    device = 0
    visible = os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)")
else:
    device = -1
    visible = "(cpu)"

print("CUDA_VISIBLE_DEVICES:", visible, "| pipeline device:", device, "| cuda_count:", torch.cuda.device_count())

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("CUDA:", torch.cuda.is_available(), "| device:", device)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=dtype)
if torch.cuda.is_available():
    model = model.to("cuda")

asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

print("ASR pipeline ready")

In [ ]:
audio_files = sorted(glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.flac"), recursive=True))
if MAX_FILES > 0:
    audio_files = audio_files[:MAX_FILES]

print("# audio files:", len(audio_files))
audio_files[:5]

In [ ]:
def transcribe_chunked(files, batch_size=8, step=10):
    all_rows = []
    t0 = time.time()
    for start in range(0, len(files), step):
        chunk = files[start:start+step]
        outs = asr(chunk, generate_kwargs={"language": LANGUAGE, "task": TASK}, batch_size=batch_size)
        for f, out in zip(chunk, outs):
            all_rows.append({"file": str(f), "fileid": os.path.basename(f), "hyp": out["text"].strip()})
        print(f"[{min(start+step, len(files))}/{len(files)}] done")
    df = pd.DataFrame(all_rows)
    print("Elapsed(s):", round(time.time() - t0, 2))
    return df

print("음성인식을 시작합니다")
df_hyp = transcribe_chunked(audio_files, batch_size=8)
df_hyp.head()
print("음성인식을 완료했습니다")

In [ ]:
import os

df_ref = pd.read_csv(REF_CSV, sep='\t')

print("df_ref columns:", df_ref.columns.tolist())
print("df_hyp columns:", df_hyp.columns.tolist())

# df_ref: file 컬럼 정규화
if "file" not in df_ref.columns:
    raise ValueError("refs.csv에서 file 컬럼을 찾지 못했습니다. columns=" + str(df_ref.columns.tolist()))
df_ref["file"] = df_ref["file"].astype(str).apply(os.path.basename)

# df_ref: transcript -> ref
if "ref" not in df_ref.columns:
    if "transcript" in df_ref.columns:
        df_ref = df_ref.rename(columns={"transcript": "ref"})
    else:
        raise ValueError("refs.csv에서 정답 텍스트 컬럼을 찾지 못했습니다. columns=" + str(df_ref.columns.tolist()))
df_ref["ref"] = df_ref["ref"].astype(str)

# df_hyp: fileid가 없으면 생성
if "fileid" not in df_hyp.columns:
    df_hyp = df_hyp.copy()
    df_hyp["fileid"] = df_hyp["file"].astype(str).apply(os.path.basename)

# merge: df_ref.file (basename)  <->  df_hyp.fileid (basename)
df = df_ref.merge(df_hyp, left_on="file", right_on="fileid", how="inner", suffixes=("_ref", "_hyp"))
df["wav_path"] = df["file_hyp"]   # df_hyp의 file(전체경로)

print("merged rows:", len(df), "/", len(df_ref), "(refs)")
if df.empty:
    print("예시 refs file:", df_ref["file"].head().tolist())
    print("예시 hyp fileid:", df_hyp["fileid"].head().tolist())
    raise ValueError("refs.csv의 file 값과 df_hyp의 fileid 값이 매칭되지 않습니다. 파일명 규칙을 맞춰야 합니다.")

print("WER:", round(wer(df["ref"].tolist(), df["hyp"].tolist()), 4))
print("CER:", round(cer(df["ref"].tolist(), df["hyp"].tolist()), 4))

df.head()

In [ ]:
import os
from IPython.display import Audio, display

for _, r in df.iterrows():
    print("FILE:", r["file_ref"])
    print("REF:", r["ref"])
    print("HYP:", r["hyp"])
    print("PATH:", r["wav_path"])
    display(Audio(filename=r["wav_path"]))
    print("-" * 60)